# Fomo Social Trading Network Analysis

**Question:** Which crypto wallets on Fomo are the most influential, and which traders occupy the most central positions in Fomo's social trading network?

- **Nodes** = Fomo traders (each linked to a crypto wallet)
- **Edges** = following relationships. `A → B` means trader A follows trader B.
- **Influence** = in-degree (how many sampled traders follow you) and betweenness centrality (how often you sit on the shortest path between other traders).

## 1. Setup

Get a free API key at https://fomoapi.io/dashboard. The key is typed in when the cell runs, so it is never saved in the notebook.

In [ ]:
import os
import time
from getpass import getpass

import requests
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

BASE_URL = "https://api.fomoapi.io"
API_KEY = getpass("FOMO API key: ")
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

## 2. Collect the top 50 traders from the 30-day leaderboard

In [ ]:
response = requests.get(f"{BASE_URL}/v2/leaderboard/30d", headers=HEADERS, params={"limit": 50})
response.raise_for_status()

traders = response.json()["traders"]
print("Number of traders collected:", len(traders))

## 3. Collect who each trader follows

Every trader becomes a node. Every following relationship becomes an edge.

In [ ]:
nodes = {}
edges = []

# The leaderboard traders are nodes, with their wallets
for trader in traders:
    nodes[trader["handle"]] = {
        "handle": trader["handle"],
        "followers": trader["followers"],
        "wallet_solana": trader["wallets"].get("solana"),
        "wallet_evm": trader["wallets"].get("evm"),
    }

# Each account a trader follows is an edge (and a node, if it is new)
for trader in traders:
    handle = trader["handle"]
    response = requests.get(f"{BASE_URL}/v2/users/{handle}/following", headers=HEADERS)

    if response.status_code != 200:
        print("Could not collect:", handle)
        continue

    # The list of followed accounts is the list inside the response
    following = next((v for v in response.json().values() if isinstance(v, list)), [])

    for followed in following:
        edges.append({"source": handle, "target": followed["handle"]})
        nodes.setdefault(followed["handle"], {
            "handle": followed["handle"],
            "followers": followed.get("followers"),
        })

    time.sleep(0.2)

nodes = pd.DataFrame(nodes.values())
edges = pd.DataFrame(edges)

## 4. Validate the data

In [ ]:
print("Missing values in edges:")
print(edges.isnull().sum())

edges = edges.dropna().drop_duplicates()

print("Nodes:", len(nodes))
print("Edges:", len(edges))
edges.head()

## 5. Build the network

In [ ]:
G = nx.from_pandas_edgelist(edges, "source", "target", create_using=nx.DiGraph)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

## 6. Calculate centrality

In [ ]:
nodes["in_degree"] = nodes["handle"].map(dict(G.in_degree())).fillna(0).astype(int)
nodes["betweenness"] = nodes["handle"].map(nx.betweenness_centrality(G)).fillna(0)

nodes = nodes.sort_values(["in_degree", "betweenness"], ascending=False)
nodes.head(10)

## 7. Save the nodes and edges

In [ ]:
nodes.to_csv("data/nodes.csv", index=False)
edges.to_csv("data/edges.csv", index=False)

## 8. Visualize the network

In [ ]:
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, node_size=30, width=0.2, arrowsize=5, with_labels=False)
plt.title("Fomo Social Trading Network")
plt.axis("off")
plt.savefig("figures/fomo_network.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Most connected traders

In [ ]:
top10 = nodes.head(10)

plt.figure(figsize=(10, 6))
plt.bar(top10["handle"], top10["in_degree"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Fomo Trader")
plt.ylabel("Number of Followers in Sample")
plt.title("Most Connected Fomo Traders by In-Degree")
plt.tight_layout()
plt.savefig("figures/fomo_top_in_degree.png", dpi=300)
plt.show()